# Used Car Data Preprocessing
## Day 12 Assignment

### Introduction
This notebook performs a complete preprocessing workflow for the Used Car Resale Dataset, including inspection, outlier handling, categorical encoding, scaling, train-test splitting, and data-leakage prevention.

## Objective
1. Inspect the dataset.
2. Identify features and target.
3. Check missing values and duplicates.
4. Handle outliers using IQR.
5. Encode categorical variables.
6. Scale numerical features.
7. Split train/test data.
8. Fit transformations only on training data.
9. Verify and save processed data.

## Dataset Description
The dataset contains used-car information including brand, year, mileage, engine capacity, power, fuel type, transmission, city, seller type, condition, ownership history, accidents, service score, and resale price.

**Target:** `Resale_Price_Lakh`.

## Step 1: Import Required Libraries
Pandas and NumPy are used for data manipulation, while Scikit-learn is used for splitting and preprocessing.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
pd.set_option("display.max_columns", None)

## Step 2: Upload the Dataset
Upload the CSV file from your local device into Google Colab.

In [ ]:
from google.colab import files
uploaded = files.upload()

## Step 3: Load the Dataset
The uploaded CSV is loaded into a Pandas DataFrame and the first rows are displayed.

In [ ]:
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)
df.head()

## Step 4: Initial Dataset Inspection
Check shape, column names, data types, and general dataset information.

In [ ]:
print("Dataset Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\nDataset Information:")
df.info()

## Step 5: Check Missing Values and Duplicate Rows
Missing values and duplicate records are checked before preprocessing.

In [ ]:
print("Missing Values:\n", df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())

## Step 6: Statistical Summary
Descriptive statistics help understand numerical distributions and quartiles needed for IQR outlier detection.

In [ ]:
df.describe()

## Step 7: Separate Features and Target Variable
`Resale_Price_Lakh` is the target. `Car_ID` is removed because it is only an identifier.

In [ ]:
target = "Resale_Price_Lakh"
X = df.drop(columns=[target, "Car_ID"])
y = df[target]
print("Features:", X.shape)
print("Target:", y.shape)

## Step 8: Split Data into Training and Testing Sets
The data is split into 80% training and 20% testing **before preprocessing** to prevent data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("Training:", X_train.shape)
print("Testing:", X_test.shape)

# Outlier Detection and Handling
## Step 9: Identify Numerical Features
IQR-based outlier detection will be applied to the numerical variables.

In [ ]:
numeric_features = ["Year","Mileage_Km","Engine_CC","Power_BHP","Previous_Owners","Accidents_Reported","Service_Score"]
print(numeric_features)

## Step 10: Detect Outliers Using the IQR Method
**IQR = Q3 − Q1**

**Lower Bound = Q1 − 1.5 × IQR**

**Upper Bound = Q3 + 1.5 × IQR**

Bounds are calculated from training data only.

In [ ]:
outlier_data = []
for col in numeric_features:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((X_train[col] < lower_bound) | (X_train[col] > upper_bound)).sum()
    outlier_data.append([col,Q1,Q3,IQR,lower_bound,upper_bound,outliers])
outlier_report = pd.DataFrame(outlier_data, columns=["Feature","Q1","Q3","IQR","Lower Bound","Upper Bound","Number of Outliers"])
outlier_report

## Step 11: Handle Outliers
Extreme values are capped at IQR boundaries instead of deleting rows. The boundaries are learned during `fit()` on training data.

In [ ]:
class IQRClipper(BaseEstimator, TransformerMixin):
    def __init__(self, multiplier=1.5): self.multiplier = multiplier
    def fit(self, X, y=None):
        X = np.asarray(X, dtype=float)
        self.q1_ = np.percentile(X, 25, axis=0)
        self.q3_ = np.percentile(X, 75, axis=0)
        self.iqr_ = self.q3_ - self.q1_
        self.lower_ = self.q1_ - self.multiplier * self.iqr_
        self.upper_ = self.q3_ + self.multiplier * self.iqr_
        return self
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.lower_, self.upper_)
    def get_feature_names_out(self, input_features=None):
        return np.asarray(input_features, dtype=object)

# Categorical Data Encoding
## Step 12: Identify Nominal and Ordinal Features
Nominal features have no natural order and use One-Hot Encoding. `Condition` has a natural order and uses Ordinal Encoding.

In [ ]:
nominal_features = ["Brand","Fuel_Type","Transmission","City","Seller_Type"]
ordinal_features = ["Condition"]
print("Nominal:", nominal_features)
print("Ordinal:", ordinal_features)

## Step 13: Define the Order for Condition
The order is: **Poor → Fair → Good → Excellent**.

In [ ]:
condition_order = [["Poor","Fair","Good","Excellent"]]

# Feature Scaling and Preprocessing Pipeline
## Step 14: Create the Numerical Pipeline
Numerical data undergoes IQR outlier capping followed by `StandardScaler`.

In [ ]:
numeric_pipeline = Pipeline([("outlier_handling", IQRClipper(1.5)), ("scaling", StandardScaler())])

## Step 15: Create the Ordinal Encoding Pipeline
The ordered `Condition` categories are encoded numerically.

In [ ]:
ordinal_pipeline = Pipeline([("ordinal_encoding", OrdinalEncoder(categories=condition_order, handle_unknown="use_encoded_value", unknown_value=-1))])

## Step 16: Create the Nominal Encoding Pipeline
Nominal categorical variables are converted into binary columns using One-Hot Encoding.

In [ ]:
nominal_pipeline = Pipeline([("one_hot_encoding", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

## Step 17: Combine All Preprocessing Steps
A `ColumnTransformer` applies the correct preprocessing pipeline to each group of columns.

In [ ]:
preprocessor = ColumnTransformer([("numerical", numeric_pipeline, numeric_features),("ordinal", ordinal_pipeline, ordinal_features),("nominal", nominal_pipeline, nominal_features)])

# Apply Preprocessing Without Data Leakage
## Step 18: Fit the Preprocessor on Training Data
Use `fit_transform()` only on training data and `transform()` on testing data. This prevents information from the test set influencing preprocessing.

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print("Training and testing data processed successfully.")

## Step 19: Retrieve Processed Feature Names
Feature names are extracted after encoding.

In [ ]:
feature_names = preprocessor.get_feature_names_out()
print("Number of processed features:", len(feature_names))
print(feature_names)

## Step 20: Convert Processed Arrays into DataFrames
Processed arrays are converted into readable Pandas DataFrames.

In [ ]:
X_train_processed_df = pd.DataFrame(X_train_processed, columns=feature_names)
X_test_processed_df = pd.DataFrame(X_test_processed, columns=feature_names)
X_train_processed_df.head()

## Step 21: Add the Target Variable Back
The target is added back after feature preprocessing.

In [ ]:
train_processed = X_train_processed_df.copy()
test_processed = X_test_processed_df.copy()
train_processed[target] = y_train.values
test_processed[target] = y_test.values
display(train_processed.head())

# Dataset Verification
## Step 22: Verify the Processed Datasets
Check shapes, missing values, duplicates, and successful preprocessing.

In [ ]:
print("Train shape:", train_processed.shape)
print("Test shape:", test_processed.shape)
print("Train missing:", train_processed.isnull().sum().sum())
print("Test missing:", test_processed.isnull().sum().sum())
print("Train duplicates:", train_processed.duplicated().sum())
print("Test duplicates:", test_processed.duplicated().sum())

## Step 23: Verify Numerical Feature Scaling
Training numerical features should be centered approximately around zero after standardization.

In [ ]:
numerical_processed_columns = ["numerical__Year","numerical__Mileage_Km","numerical__Engine_CC","numerical__Power_BHP","numerical__Previous_Owners","numerical__Accidents_Reported","numerical__Service_Score"]
train_processed[numerical_processed_columns].describe()

# Save the Preprocessed Dataset
## Step 24: Create the Complete Processed Dataset
The complete dataset is transformed using the preprocessor already fitted on training data. No new fitting is performed.

In [ ]:
X_processed = preprocessor.transform(X)
processed_df = pd.DataFrame(X_processed, columns=feature_names)
processed_df[target] = y.values
processed_df.head()

## Step 25: Save the Processed Files
Save the complete processed dataset, train set, test set, and outlier report.

In [ ]:
processed_df.to_csv("Used_Car_Preprocessed_Dataset.csv", index=False)
train_processed.to_csv("Used_Car_Preprocessed_Train.csv", index=False)
test_processed.to_csv("Used_Car_Preprocessed_Test.csv", index=False)
outlier_report.to_csv("Used_Car_Training_Outlier_Report.csv", index=False)
print("All files saved successfully!")

# Final Results
The workflow completed dataset inspection, train/test splitting, IQR outlier handling, ordinal and nominal encoding, numerical scaling, verification, and saving. All transformations were fitted only on training data to prevent data leakage.

# Conclusion
A complete and leakage-safe preprocessing workflow was performed. The final dataset is suitable for machine learning models that predict used car resale prices.

**Submission:** Upload this notebook and the processed dataset to GitHub, then submit the GitHub repository link in the LMS.